# WTI and ROC(12) in Plotly

Replica sencilla del gráfico `ROC(12)` usando `pandas` y `plotly`.

- Descarga WTI diario desde FRED (`DCOILWTICO`)
- Lo agrega a frecuencia mensual (`open`, `high`, `low`, `close`)
- Inserta el snapshot provisional del mes actual desde Yahoo Finance (`CL=F`)
- Calcula `ROC(12)` y arma el gráfico en dos paneles


In [3]:
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
from plotly.subplots import make_subplots
from pathlib import Path

FRED_WTI_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILWTICO"
YAHOO_WTI_SYMBOL = "CL=F"


In [4]:
df = pd.read_csv(FRED_WTI_URL)
df = df.rename(columns={"DATE": "date", "observation_date": "date", "DCOILWTICO": "price"})
df["date"] = pd.to_datetime(df["date"])
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["date", "price"])
df = df[df["price"] > 0].sort_values("date").reset_index(drop=True)

monthly = (
    df.assign(month=df["date"].dt.to_period("M").dt.to_timestamp())
      .groupby("month", as_index=False)
      .agg(
          open=("price", "first"),
          high=("price", "max"),
          low=("price", "min"),
          close=("price", "last"),
      )
      .rename(columns={"month": "date"})
)

today = pd.Timestamp.today().normalize()
current_month_start = today.to_period("M").to_timestamp()
recent = yf.download(
    YAHOO_WTI_SYMBOL,
    start=current_month_start.strftime("%Y-%m-%d"),
    end=(today + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    auto_adjust=False,
    progress=False,
)

if isinstance(recent.columns, pd.MultiIndex):
    recent.columns = recent.columns.get_level_values(0)

recent = recent.rename(columns=str.title)
recent = recent[["Open", "High", "Low", "Close"]].dropna().reset_index()
recent["Date"] = pd.to_datetime(recent["Date"])

snapshot = pd.DataFrame([
    {
        "date": current_month_start,
        "open": float(recent.loc[0, "Open"]),
        "high": float(recent["High"].max()),
        "low": float(recent["Low"].min()),
        "close": float(recent.loc[len(recent) - 1, "Close"]),
    }
])
monthly = monthly[monthly["date"] != snapshot.loc[0, "date"]]
monthly = pd.concat([monthly, snapshot], ignore_index=True).sort_values("date").reset_index(drop=True)
monthly["roc12"] = ((monthly["close"] / monthly["close"].shift(12)) - 1) * 100
monthly.tail()


,date,open,high,low,close,roc12
479,2025-12-01,59.470000,60.230000,55.44,57.260000,-20.955273
480,2026-01-01,57.210000,64.770000,56.01,64.500000,-11.449753
481,2026-02-01,61.600000,66.960000,61.60,66.960000,-4.301844
482,2026-03-01,71.130000,104.690000,71.13,104.690000,45.665785
483,2026-04-01,101.720001,115.480003,96.50,112.440002,88.816125


In [7]:
events = [
    ("1987 Crash", "1987-10-01"),
    ("1990 Crash", "1990-08-01"),
    ("DOT COM", "2000-03-01"),
    ("Financial Crisis", "2008-10-01"),
    ("2022 Bear Market<br>Inflation / War / Rates", "2022-06-01"),
]
events_df = pd.DataFrame(events, columns=["label", "date"])
events_df["date"] = pd.to_datetime(events_df["date"])
events_df["roc12"] = events_df["date"].map(monthly.set_index("date")["roc12"])
events_df


,label,date,roc12
0,1987 Crash,1987-10-01,30.885246
1,1990 Crash,1990-08-01,45.778014
2,DOT COM,2000-03-01,61.224490
3,Financial Crisis,2008-10-01,-27.676296
4,2022 Bear Market<br>Inflation / War / Rates,2022-06-01,46.572361


In [6]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    row_heights=[0.7, 0.3],
)

fig.add_trace(
    go.Candlestick(
        x=monthly["date"],
        open=monthly["open"],
        high=monthly["high"],
        low=monthly["low"],
        close=monthly["close"],
        increasing_line_color="#00c46a",
        increasing_fillcolor="#00c46a",
        decreasing_line_color="#ef2f2f",
        decreasing_fillcolor="#ef2f2f",
        whiskerwidth=0.4,
        name="WTI",
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=monthly["date"],
        y=monthly["roc12"],
        mode="lines",
        line=dict(color="#ffd166", width=3),
        name="ROC(12)",
        hovertemplate="%{x|%b %Y}<br>ROC(12): %{y:.2f}%<extra></extra>",
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig.add_hline(y=100, line_color="rgba(255,255,255,0.45)", line_width=1, row=2, col=1)
fig.add_vrect(
    x0="2026-01-01",
    x1="2028-12-31",
    fillcolor="rgba(255,96,80,0.16)",
    line_width=0,
    row=2,
    col=1,
)

fig.add_annotation(
    x=pd.Timestamp("2027-01-01"),
    y=175,
    text="Rally<br>2026",
    showarrow=False,
    font=dict(color="#385dff", size=16, family="Arial"),
    row=2,
    col=1,
)

for _, event in events_df.dropna(subset=["roc12"]).iterrows():
    fig.add_annotation(
        x=event["date"],
        y=float(event["roc12"]),
        text=event["label"],
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2.5,
        arrowcolor="#385dff",
        ax=0,
        ay=-55,
        font=dict(color="#385dff", size=12),
        align="center",
        row=2,
        col=1,
    )

fig.update_layout(
    template="plotly_dark",
    title="WTI: precios y momentum en niveles historicamente elevados",
    paper_bgcolor="#050505",
    plot_bgcolor="#050505",
    font=dict(family="Arial", color="#f2f2f2"),
    margin=dict(l=62, r=30, t=80, b=80),
    xaxis_rangeslider_visible=False,
    hovermode="x unified",
    annotations=list(fig.layout.annotations) + [
        dict(
            x=0,
            y=-0.18,
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            font=dict(size=12, color="rgba(220,220,220,0.82)"),
            text=(
                "Nota: El panel superior muestra precios mensuales del WTI: en verde cuando el precio de cierre "
                "es mayor que el de apertura y en rojo cuando es menor. Cada barra mensual usa como precio de "
                "apertura el primer precio observado del mes y como precio de cierre el ultimo. Para el mes en curso, "
                "el grafico usa un snapshot provisional desde Yahoo Finance para capturar el movimiento actual. "
                "El panel inferior muestra ROC = (p_t - p_t-12) / p_t-12 * 100, donde p es el precio de cierre del dia."
            ),
        ),
        dict(
            x=0,
            y=-0.28,
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            font=dict(size=12, color="rgba(220,220,220,0.82)"),
            text="Fuente: FRED DCOILWTICO; Ted @TedPillows, @marketmike.",
        ),
    ],
)

fig.update_xaxes(
    showgrid=False,
    tickformat="%y",
    dtick="M12",
)
fig.update_yaxes(showgrid=False, zeroline=False, row=1, col=1)
fig.update_yaxes(showgrid=False, zeroline=False, row=2, col=1)

fig.show()


In [ ]:
# Opcional: guardar HTML interactivo al lado del notebook
out = Path("wti_roc12_plotly.html")
fig.write_html(out, include_plotlyjs="cdn")
out
